In [ ]:
import ast
import math
from collections import Counter
from igraph import Graph
from itertools import combinations


#############################################################
# RARITY OF A RULE
#############################################################

def rarity_rule(rule, global_frequency):

    rarity = 0

    for pair in rule:

        rarity += global_frequency[pair]

    return 1- (rarity / len(rule))

#############################################################
# GLOBAL ATTRIBUTE-VALUE FREQUENCY
#############################################################

from collections import Counter

def compute_global_frequency(rules):

    counter = Counter()

    total_pairs = 0

    for rule in rules:

        for pair in rule:

            counter[pair] += 1
            total_pairs += 1

    freq = {}

    for pair, count in counter.items():

        freq[pair] = count / total_pairs

    return freq

#############################################################
# REUSE OF A RULE
#############################################################

def reuse_rule(rule,
               community_id,
               community_frequency):

    reuse = 0

    freq = community_frequency[community_id]

    for pair in rule:

        reuse += freq[pair]

    return 1 - (reuse / len(rule))

#############################################################
# COMMUNITY ATTRIBUTE-VALUE FREQUENCY
#############################################################

from collections import defaultdict

def compute_community_frequency(rules, membership):

    community_rules = defaultdict(list)

    # Agrupar reglas por comunidad
    for idx, comm in enumerate(membership):

        community_rules[comm].append(idx)

    community_frequency = {}

    for comm, rule_ids in community_rules.items():

        counter = Counter()

        n_rules = len(rule_ids)

        for rid in rule_ids:

            # Para evitar contar dos veces el mismo atributo si aparece repetido
            for pair in set(rules[rid]):

                counter[pair] += 1

        freq = {}

        for pair, count in counter.items():

            freq[pair] = count / n_rules

        community_frequency[comm] = freq

    return community_frequency

def jaccard_similarity(rule1, rule2):

    s1 = set(rule1)
    s2 = set(rule2)

    inter = len(s1.intersection(s2))
    union = len(s1.union(s2))

    if union == 0:
        return 0

    return inter / union

#############################################################
# NKC METRIC
#############################################################

def compute_NKC(rules,membership,alpha=1):

    global_frequency = compute_global_frequency(

        rules

    )

    community_frequency = compute_community_frequency(

        rules,

        membership

    )

    nkc = 0
    nkc_reuse = 0
    nkc_rarity = 0

    details = []

    for idx, rule in enumerate(rules):

        size = len(rule)

        rarity = rarity_rule(

            rule,

            global_frequency

        )

        reuse = reuse_rule(

            rule,

            membership[idx],

            community_frequency

        )

        if reuse == 0:
            reuse = 1/len(rule)

        score = (
            size ** alpha
        ) *  (
            (rarity + reuse) / 2
        )

        nkc += score
        nkc_reuse += ( size ** alpha ) * ( reuse )
        nkc_rarity += ( size ** alpha ) * ( rarity )

        details.append({

            "rule": idx,

            "community": membership[idx],

            "size": size,

            "rarity": rarity,

            "reuse": reuse,

            "complexity": score

        })

    return nkc, details, nkc_reuse, nkc_rarity

#############################################################
# COMMUNITY DETECTION
#############################################################

def detect_communities(graph):

    communities = graph.community_multilevel(
        weights=graph.es["weight"]
    )

    graph.vs["community"] = communities.membership

    return communities

#############################################################
# RULE GRAPH
#############################################################

def build_rule_graph(rules, threshold=0.5):

    g = Graph()

    g.add_vertices(len(rules))

    edges = []
    weights = []

    for i, j in combinations(range(len(rules)), 2):

        sim = jaccard_similarity(
            rules[i],
            rules[j]
        )

        if sim >= threshold:

            edges.append((i, j))
            weights.append(sim)

    g.add_edges(edges)

    g.es["weight"] = weights

    return g

def load_rules(file_name):

    rules = []

    with open(file_name, "r") as f:

        for line in f:

            line = line.strip()

            if line == "":
                continue

            rule = ast.literal_eval(line)

            attrs = []

            for att in rule[1]:

                attrs.append(
                    (att[0], str(att[1]))
                )

            rules.append(attrs)

    return rules


#############################################################
# WSC
#############################################################

def compute_wsc(rules):

    return sum(len(rule) for rule in rules)


#############################################################
# GLOBAL FREQUENCIES
#############################################################

def compute_global_frequencies(rules):

    counter = Counter()

    total_pairs = 0

    for rule in rules:

        for pair in rule:

            counter[pair] += 1
            total_pairs += 1

    frequencies = {}

    for pair, count in counter.items():

        frequencies[pair] = count / total_pairs

    return frequencies

#############################################################
# COMMUNITY REUSE FACTOR
#############################################################

def compute_reuse_factor(rules, membership):

    reuse = {}

    n_communities = max(membership) + 1

    for cid in range(n_communities):

        unique_pairs = set()

        total_pairs = 0

        for idx, rule in enumerate(rules):

            if membership[idx] != cid:
                continue

            total_pairs += len(rule)

            for pair in rule:
                unique_pairs.add(pair)

        if total_pairs == 0:

            reuse[cid] = 1.0

        else:

            reuse[cid] = len(unique_pairs) / total_pairs

    return reuse

#############################################################
# SHANNON RARITY
#############################################################

def rarity_rule(rule, frequencies):

    rarity = 0

    for pair in rule:

        rarity += frequencies[pair]

    return rarity / len(rule)


#############################################################
# NEW METRIC
#############################################################



#############################################################
# PRINT COMPARISON
#############################################################

def compare_metrics(file_name, edge_th, alpha_):

    rules = load_rules(file_name)

    ####################################################
    # RULE GRAPH
    ####################################################

    graph = build_rule_graph(
        rules,
        threshold=edge_th
    )

    ####################################################
    # COMMUNITY DETECTION
    ####################################################

    communities = detect_communities(graph)
    print("\nRule Graph")
    print("Nodes:", graph.vcount())
    print("Edges:", graph.ecount())
    print("Density:", round(graph.density(),4))

    print("Communities:", len(communities))

    sizes = [len(c) for c in communities]

    print("Community sizes:", sizes)

    wsc = compute_wsc(rules)

    nkc, details, nkc_reuse, nkc_rarity = compute_NKC(
        rules,
        communities.membership,
        alpha=alpha_
    )


    print("="*70)

    print("Rule | Comm | Size | Rareza | Reuso | NKC")

    print("="*70)

    for d in details:

        print(

            f"{d['rule']:3d} | "

            f"{d['community']:3d} | "

            f"{d['size']:2d} | "

            f"{d['rarity']:.3f} | "

            f"{d['reuse']:.3f} | "

            f"{d['complexity']:.3f}"

    )
    
    print("=" * 60)

    print("Policy")

    print("=" * 60)

    print("Rules:", len(rules))

    print("WSC:", round(wsc, 2))
    
    print(f"NKC(G_R): {nkc:.4f}")
    print(f"NKC(G_R)-Rarity: {nkc_rarity:.4f}")
    print(f"NKC(G_R)-Reuse: {nkc_reuse:.4f}")

    #Comparing

    comparing = (wsc-nkc) / wsc

    print(f"DIFF: {comparing:.4f}")

    #compare_metrics("/home/daniel/Documents/phd/phd-thesis-lab/17_NuevoAnalisisP1/reglas_abac.txt")

In [161]:
compare_metrics("policy2.txt", 0.25, 1.2)


Rule Graph
Nodes: 5
Edges: 0
Density: 0.0
Communities: 5
Community sizes: [1, 1, 1, 1, 1]
Rule | Comm | Size | Rareza | Reuso | NKC
  0 |   0 |  4 | 0.050 | 0.000 | 0.132
  1 |   1 |  4 | 0.050 | 0.000 | 0.132
  2 |   2 |  4 | 0.050 | 0.000 | 0.132
  3 |   3 |  4 | 0.050 | 0.000 | 0.132
  4 |   4 |  4 | 0.050 | 0.000 | 0.132
Policy
Rules: 5
WSC: 20
NKC(G_R): 0.6598
NKC(G_R)-Rarity: 1.3195
NKC(G_R)-Reuse: 0.0000
DIFF: 0.9670
